In [15]:
import pandas as pd
path = r"C:\Users\papan\Desktop\Coffeomics\proteomics\de_raw_results\de_raw_results_50_above10\de_raw_results.xlsx"
df = pd.read_excel(path, index_col=0, header=[0,1,2,3,4]) #sheet_name="log2FoldChange"
df

medium    380ppm             25/20°C            \
               severe    700ppm             37/28°C   42/30°C   
              25/20°C   25/20°C              380ppm    380ppm   
               380ppm    medium    severe    severe    severe   
protein     C.arabica C.arabica C.arabica C.arabica C.arabica   
Cc00_g00190                                                     
Cc00_g04970       NaN       NaN       NaN       NaN       NaN   
Cc00_g05110       NaN       NaN       NaN       NaN       NaN   
Cc00_g07470       NaN       NaN       NaN       NaN       NaN   
Cc00_g09510       NaN       NaN       NaN       NaN       NaN   
Cc00_g11300       NaN       NaN       NaN       NaN       NaN   
...               ...       ...       ...       ...       ...   
Cc11_g14730       NaN       NaN       NaN       NaN       NaN   
Cc11_g15580       NaN       NaN       NaN       NaN       NaN   
Cc11_g16360       NaN       NaN       NaN       NaN       NaN   
Cc11_g16510       NaN       NaN       NaN       NaN       NaN   
Cc11_g17290       NaN       NaN       NaN       NaN       NaN   

                               without              380ppm   25/20°C  ...  \
            Rec/14/-/25/20°C    medium    severe    700ppm   37/28°C  ...   
                      380ppm   25/20°C   25/20°C   25/20°C    380ppm  ...   
                      severe    380ppm    380ppm   without   without  ...   
protein            C.arabica C.arabica C.arabica C.arabica C.arabica  ...   
Cc00_g00190                                                           ...   
Cc00_g04970              NaN       NaN       NaN       NaN       NaN  ...   
Cc00_g05110              NaN       NaN       NaN       NaN       NaN  ...   
Cc00_g07470              NaN       NaN       NaN       NaN       NaN  ...   
Cc00_g09510              NaN       NaN       NaN       NaN       NaN  ...   
Cc00_g11300              NaN       NaN       NaN       NaN       NaN  ...   
...                      ...       ...       ...       ...       ...  ...   
Cc11_g14730              NaN       NaN       NaN       NaN       NaN  ...   
Cc11_g15580              NaN       NaN       NaN       NaN       NaN  ...   
Cc11_g16360              NaN       NaN       NaN       NaN       NaN  ...   
Cc11_g16510              NaN       NaN       NaN       NaN       NaN  ...   
Cc11_g17290              NaN       NaN       NaN       NaN       NaN  ...   

                without      380ppm          42/30°C                 without  \
                 severe      700ppm Rec/14/-/25/20°C                  severe   
                42/30°C     42/30°C           380ppm      700ppm     42/30°C   
                 380ppm     without          without      severe      700ppm   
protein     C.canephora C.canephora      C.canephora C.canephora C.canephora   
Cc00_g00190                                                                    
Cc00_g04970         NaN         NaN              NaN         NaN         NaN   
Cc00_g05110         NaN         NaN              NaN         NaN         NaN   
Cc00_g07470         NaN         NaN              NaN         NaN         NaN   
Cc00_g09510         NaN         NaN              NaN         NaN         NaN   
Cc00_g11300         NaN         NaN              NaN         NaN         NaN   
...                 ...         ...              ...         ...         ...   
Cc11_g14730         NaN         NaN              NaN         NaN         NaN   
Cc11_g15580         NaN         NaN              NaN         NaN         NaN   
Cc11_g16360         NaN         NaN              NaN         NaN         NaN   
Cc11_g16510         NaN         NaN              NaN         NaN         NaN   
Cc11_g17290         NaN         NaN              NaN         NaN         NaN   

                     42/30°C           380ppm          without  \
            Rec/14/-/25/20°C           700ppm           severe   
                      700ppm Rec/14/-/25/20°C Rec/14/-/25/20°C   
                     without           s

In [4]:
import pandas as pd
import logging
from pathlib import Path

logger = logging.getLogger(__name__)

# Variation name → row-selection predicate applied per condition column.
# NaN values evaluate to False for both > 0 and < 0, so they are
# automatically excluded from positive/negative without extra handling.
_VARIATION_MASKS = {
    "non_nan":  lambda col: col.notna(),
    "positive": lambda col: col > 0,
    "negative": lambda col: col < 0,
}


def df_to_prep_inputs(
    df: pd.DataFrame,
    output_dir: str | Path,
    sp: str | None = None,
    transcript_col: str | None = None,
    col_sep: str = "__",
) -> dict[str, dict[str, int]]:
    """
    Write one CSV per condition for each of three variation sub-folders
    (non_nan, positive, negative) so that prep() can consume them directly.

    Output layout (mirrors INPUT_DATA structure expected by prep()):
        output_dir / [sp /] {non_nan|positive|negative} / <condition>.csv

    Variation semantics (applied per condition column, per row):
        non_nan  – value is not NaN                  (all DE genes)
        positive – value > 0                         (up-regulated)
        negative – value < 0                         (down-regulated)

    Parameters
    ----------
    df : pd.DataFrame
        - If *transcript_col* is None (default): the DataFrame index must hold
          transcript/gene identifiers; every column is treated as a condition.
        - If *transcript_col* is given: that column holds identifiers; all
          remaining columns are treated as conditions.
    output_dir : str or Path
        Root directory for generated files. When *sp* is supplied the final
        path becomes ``output_dir / sp / {variation}``.
    sp : str, optional
        Species subfolder (mirrors prep()'s ``sp`` argument).
    transcript_col : str, optional
        Name of the column holding transcript IDs when they are not the index.
    col_sep : str
        Separator used to flatten MultiIndex column tuples (default ``"__"``).

    Returns
    -------
    dict[str, dict[str, int]]
        ``{ variation: { condition_label: n_transcripts_written } }``
    """
    output_dir = Path(output_dir)
    if sp is not None:
        output_dir = output_dir / sp

    # ── 1. Separate identifiers from condition columns ───────────────────────
    if transcript_col is not None:
        if transcript_col not in df.columns:
            raise KeyError(f"transcript_col '{transcript_col}' not found in df.columns")
        id_series    = df[transcript_col].reset_index(drop=True)
        condition_df = df.drop(columns=[transcript_col]).reset_index(drop=True)
    else:
        id_series    = df.index.to_series()   # preserves index alignment
        condition_df = df

    # ── 2. Drop rows that are ALL NaN across every condition column ──────────
    all_nan_mask = condition_df.isna().all(axis=1)
    n_dropped    = int(all_nan_mask.sum())
    condition_df = condition_df.loc[~all_nan_mask]
    id_series    = id_series.loc[~all_nan_mask]

    logger.info(
        "All-NaN row removal: %d dropped, %d retained (%.1f%% kept).",
        n_dropped,
        len(condition_df),
        100 * len(condition_df) / max(len(df), 1),
    )

    # ── 3. Pre-create all variation directories ──────────────────────────────
    var_dirs = {}
    for variation in _VARIATION_MASKS:
        var_dir = output_dir / variation
        var_dir.mkdir(parents=True, exist_ok=True)
        var_dirs[variation] = var_dir

    # ── 4. Write one CSV per condition × variation ───────────────────────────
    results: dict[str, dict[str, int]] = {v: {} for v in _VARIATION_MASKS}

    for col in condition_df.columns:
        # Flatten MultiIndex tuples to a single string label
        col_label = (
            col_sep.join(str(lvl) for lvl in col)
            if isinstance(col, tuple)
            else str(col)
        )
        col_label = col_label.replace("/", "-")   # avoid Windows path-separator issues

        col_data = condition_df[col]

        for variation, mask_fn in _VARIATION_MASKS.items():
            valid_mask  = mask_fn(col_data)
            transcripts = id_series.loc[valid_mask].values

            out_path = var_dirs[variation] / f"{col_label}.csv"
            pd.DataFrame(transcripts, columns=["transcript"]).to_csv(
                out_path, header=True, index=False
            )

            results[variation][col_label] = len(transcripts)
            logger.debug(
                "  [%-8s] %-60s → %d transcript(s) → %s",
                variation, col_label, len(transcripts), out_path,
            )

    # ── 5. Print summary ─────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  df_to_prep_inputs summary  —  species: {sp or '(none)'}")
    print(f"{'='*55}")
    for variation, written in results.items():
        n_total    = len(written)
        n_nonempty = sum(1 for count in written.values() if count > 0)
        n_empty    = n_total - n_nonempty
        print(
            f"  {variation:<10}  {n_total:>3} condition(s)  |  "
            f"{n_nonempty:>3} with genes  |  {n_empty:>3} empty"
        )
        logger.info(
            "  [%s] Generated %d condition file(s) in '%s'.",
            variation, n_total, var_dirs[variation],
        )
    print(f"{'='*55}\n")

    return results

In [17]:

df_canephora = df.xs("C.canephora", axis=1, level=4)
df_arabica   = df.xs("C.arabica",   axis=1, level=4)

for sp, sp_df in [("canephora", df_canephora), ("arabica", df_arabica)]:
    df_to_prep_inputs(
        df=sp_df,
        output_dir=r"C:\Users\papan\Desktop\Coffeomics\layers_analysis\go_by_comparison\proteomics\inputs\comparison_gene_lists",
        sp=sp,
        transcript_col=None,
        col_sep="__",
    )


  df_to_prep_inputs summary  —  species: canephora
  non_nan      45 condition(s)  |    8 with genes  |   37 empty
  positive     45 condition(s)  |    5 with genes  |   40 empty
  negative     45 condition(s)  |    4 with genes  |   41 empty


  df_to_prep_inputs summary  —  species: arabica
  non_nan      45 condition(s)  |   13 with genes  |   32 empty
  positive     45 condition(s)  |    8 with genes  |   37 empty
  negative     45 condition(s)  |   11 with genes  |   34 empty

